# A220119 — upload assets, then check with Lean

**The Lean proof remains unverified.**

1. Run the first code cell and upload the entire updated **A220119_Lean_Colab.zip**.
2. Run the remaining cells. The existing Lean/mathlib installation and caches are reused.
3. The last cell downloads sources, status, and logs, including failures.

This notebook contains no embedded archive. Select a **High-RAM CPU runtime**;
no GPU is needed. The exact minimum physical RAM requirement is unmeasured.
The runner uses compiler default memory behavior and omits `-M`/`--memory`
because of the known `max_memory` option error in the pinned Lean version.
Changed existing source files are backed up before the uploaded versions are installed.

The runner keeps mathlib's working directory for dependency paths and explicitly passes
the uploaded source directory as Lean's package root. An import-only smoke check runs
before proof modules. Success requires compilation of all modules and the exact-statement
and axiom audits.


In [ ]:
from pathlib import Path, PurePosixPath
from google.colab import files
import io, zipfile, json, hashlib, os, subprocess, time, shutil

BASE = Path('/content/A220119_Lean')
BASE.mkdir(parents=True, exist_ok=True)
SOURCES = BASE / 'source'
SOURCES.mkdir(exist_ok=True)
LOGS = BASE / 'setup_logs'
LOGS.mkdir(exist_ok=True)
STATUS = SOURCES / 'status.json'
SETUP_OK = False
STATUS.write_text(json.dumps({'verified': False, 'state': 'NOT_VERIFIED',
                             'reason': 'Starting a new upload/check run.'}, indent=2))

uploaded = files.upload()  # Select the entire updated A220119_Lean_Colab.zip.
if len(uploaded) != 1:
    raise ValueError('Upload exactly one complete A220119_Lean_Colab.zip.')
upload_name, archive_bytes = next(iter(uploaded.items()))
incoming = {}
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    if archive.testzip() is not None:
        raise ValueError('The uploaded ZIP is damaged.')
    for item in archive.infolist():
        rel = PurePosixPath(item.filename)
        if rel.is_absolute() or '..' in rel.parts:
            raise ValueError('Invalid archive path.')
        if item.is_dir() or not rel.parts or rel.parts[0] != 'A220119_Lean':
            continue
        if len(rel.parts) < 2:
            continue
        name = str(PurePosixPath(*rel.parts[1:]))
        if name in incoming:
            raise ValueError('Duplicate archive member: ' + name)
        incoming[name] = archive.read(item.filename)

required = ['compile_in_lake.py', 'lean-toolchain', 'source_manifest.json',
            'A220119/Spec.lean', 'A220119/Arithmetic.lean',
            'A220119/PolynomialTools.lean', 'A220119/Elimination.lean',
            'A220119/Main.lean', 'Audit.lean']
missing = [name for name in required if name not in incoming]
if missing:
    raise ValueError('Incomplete asset ZIP: ' + ', '.join(missing))
expected_runner = '74ba176b14b94339f0b61bbbfad432d6c5e935aa3db1093d678c80345659f478'
if hashlib.sha256(incoming['compile_in_lake.py']).hexdigest() != expected_runner:
    raise ValueError('Runner version mismatch. Upload the updated ZIP supplied with this notebook.')
manifest = json.loads(incoming['source_manifest.json'])
for name in required:
    if name.endswith('.lean'):
        if hashlib.sha256(incoming[name]).hexdigest() != manifest.get(name):
            raise ValueError('Proof source hash mismatch: ' + name)

backup = BASE / 'upload_backups' / str(time.time_ns())
for name, data in incoming.items():
    target = SOURCES / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_bytes() != data:
        prior = backup / name
        prior.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, prior)
    target.write_bytes(data)
print('Uploaded assets installed:', SOURCES)
if backup.exists(): print('Previous versions backed up:', backup)
print('Existing mathlib installation and dependency caches retained.')
print('Status: NOT_VERIFIED until the Lean build and audits complete.')


In [ ]:
# Idempotent, separate Lean/mathlib installation.
import urllib.request

SETUP_OK = False
MATHLIB_TAG = 'v4.19.0'
TOOLCHAIN = 'leanprover/lean4:v4.19.0'
MATHLIB = BASE / ('mathlib-' + MATHLIB_TAG)
env = dict(os.environ)
env['PATH'] = str(Path.home() / '.elan' / 'bin') + os.pathsep + env.get('PATH', '')
env['NO_COLOR'] = '1'
env['TERM'] = 'dumb'

def logged(command, cwd, filename, seconds=1200):
    path = LOGS / filename
    print('RUN:', ' '.join(map(str, command)), flush=True)
    # timeout supervises only this command and its child process group.
    process = subprocess.Popen(['timeout', '--signal=TERM', str(seconds)] + list(map(str,command)),
                               cwd=str(cwd), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, errors='replace')
    with path.open('w') as out:
        for line in process.stdout:
            print(line, end='', flush=True)
            out.write(line); out.flush()
    code = process.wait()
    if code:
        raise RuntimeError(f'Exit {code}; see {path}')
    return path.read_text()

try:
    elan = Path.home() / '.elan' / 'bin' / 'elan'
    if not elan.exists():
        installer = BASE / 'elan-init.sh'
        url = 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh'
        with urllib.request.urlopen(url, timeout=60) as response:
            installer.write_bytes(response.read())
        (BASE / 'elan-installer.sha256').write_text(hashlib.sha256(installer.read_bytes()).hexdigest())
        logged(['sh', installer, '-y', '--no-modify-path', '--default-toolchain', 'none'],
               BASE, 'elan_install.log')
    logged([elan, 'toolchain', 'install', TOOLCHAIN], BASE, 'toolchain_install.log')
    if not (MATHLIB / '.git').exists():
        if MATHLIB.exists() and any(MATHLIB.iterdir()):
            raise RuntimeError('Refusing to overwrite an existing non-Git directory')
        logged(['git', 'clone', '--depth', '1', '--branch', MATHLIB_TAG,
                'https://github.com/leanprover-community/mathlib4.git', MATHLIB],
               BASE, 'mathlib_clone.log')
    head = logged(['git', 'rev-parse', 'HEAD'], MATHLIB, 'mathlib_commit.log').strip()
    tag = logged(['git', 'rev-parse', MATHLIB_TAG], MATHLIB, 'mathlib_tag.log').strip()
    if head != tag:
        raise RuntimeError('Existing mathlib checkout is not at the required release; preserved unchanged.')
    toolchain = (MATHLIB / 'lean-toolchain').read_text().strip()
    if toolchain != TOOLCHAIN:
        raise RuntimeError(f'Toolchain mismatch: {toolchain}')
    logged(['lake', 'exe', 'cache', 'get'], MATHLIB, 'mathlib_cache.log', seconds=1800)
    versions = {'mathlib_tag': MATHLIB_TAG, 'mathlib_commit': head, 'toolchain': toolchain,
                'bootstrap_installer_url': 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh'}
    (BASE / 'environment.json').write_text(json.dumps(versions, indent=2))
    SETUP_OK = True
    print('Dependency setup complete. The theorem is still NOT_VERIFIED.')
except Exception as error:
    STATUS.write_text(json.dumps({'verified': False, 'state': 'NOT_VERIFIED',
                                 'setup_error': str(error)}, indent=2))
    print('SETUP FAILED:', error)


In [ ]:
# Compilation is done by real Lean. No finite Python test can pass this cell.
if SETUP_OK:
    try:
        logged(['lake', 'env', 'python3', SOURCES / 'compile_in_lake.py'],
               MATHLIB, 'kernel_build.log', seconds=1800)
    except Exception as error:
        print('NOT VERIFIED:', error)
        if not STATUS.exists():
            STATUS.write_text(json.dumps({'verified':False, 'state':'NOT_VERIFIED',
                                         'error':str(error)}, indent=2))
print(STATUS.read_text())


In [ ]:
# This cell also works after a failed proof build.
from google.colab import files
result_zip = BASE / 'A220119_Lean_run_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for parent in [SOURCES, LOGS]:
        if parent.exists():
            for path in parent.rglob('*'):
                if path.is_file() and path.suffix not in ['.olean', '.ilean']:
                    archive.write(path, str(path.relative_to(BASE)))
    for name in ['environment.json', 'elan-installer.sha256']:
        path = BASE / name
        if path.exists(): archive.write(path, name)
print('Result:', json.loads(STATUS.read_text()).get('state', 'NOT_VERIFIED'))
files.download(str(result_zip))
